## Build Gold Dimensionser

# Read sil;ver layer data

In [0]:
from pyspark.sql.functions import col

silver_base = "abfss://silver@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025"

df_customer = spark.read.parquet(
    f"{silver_base}/DimCustomer"
)

df_product = spark.read.parquet(
    f"{silver_base}/DimProduct"
)

df_date = spark.read.parquet(
    f"{silver_base}/DimDate"
)

df_geography = spark.read.parquet(
    f"{silver_base}/DimGeography"
)

df_fact = spark.read.parquet(
    f"{silver_base}/FactInternetSales"
)

print("Silver data loaded successfully")

#  create the four Gold dimensions

In [0]:
gold_dim_customer = df_customer

gold_dim_product = df_product

gold_dim_date = df_date

gold_dim_geography = df_geography

print("Gold dimension DataFrames created")

## Prepare Gold dimcustomer

In [0]:
from pyspark.sql.functions import col

gold_dim_customer = df_customer.select(
    "CustomerKey",
    "CustomerAlternateKey",
    "FullName",
    "BirthDate",
    "MaritalStatus",
    "Gender",
    "YearlyIncome",
    "TotalChildren",
    "NumberChildrenAtHome",
    "HouseOwnerFlag",
    "NumberCarsOwned",
    "DateFirstPurchase",
    "GeographyKey"
)

display(gold_dim_customer)

In [0]:
print("DimCustomer rows:", gold_dim_customer.count())
print("DimCustomer columns:", len(gold_dim_customer.columns))

gold_dim_customer.printSchema()

## Prepare Gold Dimproduct

In [0]:
gold_dim_product = df_product.select(
    "ProductKey",
    "ProductAlternateKey",
    "EnglishProductName",
    "ProductSubcategoryKey",
    "Color",
    "SafetyStockLevel",
    "ReorderPoint",
    "ListPrice",
    "StandardCost",
    "DealerPrice",
    "Weight",
    "WeightUnitMeasureCode",
    "Size",
    "SizeUnitMeasureCode",
    "SizeRange",
    "DaysToManufacture",
    "ProductLine",
    "Class",
    "Style",
    "ModelName",
    "FinishedGoodsFlag",
    "StartDate",
    "EndDate",
    "Status"
)

display(gold_dim_product.limit(10))

In [0]:
print("DimProduct rows:", gold_dim_product.count())
print("DimProduct columns:", len(gold_dim_product.columns))

gold_dim_product.printSchema()

## Build Gold DimGeography

In [0]:
gold_dim_geography = df_geography.select(
    "GeographyKey",
    "City",
    "StateProvinceCode",
    "StateProvinceName",
    "CountryRegionCode",
    "EnglishCountryRegionName",
    "PostalCode",
    "SalesTerritoryKey"
)

display(gold_dim_geography.limit(10))
print("DimGeography rows:", gold_dim_geography.count())
print("DimGeography columns:", len(gold_dim_geography.columns))

gold_dim_geography.printSchema()

## # Build DimDate

In [0]:
gold_dim_date = df_date.select(
    "DateKey",
    "FullDateAlternateKey",
    "DayNumberOfWeek",
    "EnglishDayNameOfWeek",
    "SpanishDayNameOfWeek",
    "FrenchDayNameOfWeek",
    "DayNumberOfMonth",
    "DayNumberOfYear",
    "WeekNumberOfYear",
    "EnglishMonthName",
    "SpanishMonthName",
    "FrenchMonthName",
    "MonthNumberOfYear",
    "CalendarQuarter",
    "CalendarYear",
    "CalendarSemester",
    "FiscalQuarter",
    "FiscalYear",
    "FiscalSemester"
)

display(gold_dim_date.limit(10))

In [0]:
print("DimDate rows:", gold_dim_date.count())
print("DimDate columns:", len(gold_dim_date.columns))

gold_dim_date.printSchema()

# SCD Type 2.

## SCD Type 2 for DimCustomer

In [0]:
from pyspark.sql.functions import current_date, lit

gold_dim_customer_scd = (
    gold_dim_customer
    .withColumn("EffectiveStartDate", current_date())
    .withColumn("EffectiveEndDate", lit(None).cast("date"))
    .withColumn("IsCurrent", lit(True))
)

display(gold_dim_customer_scd.limit(10))

In [0]:
gold_dim_customer_scd.printSchema()

### Validate

In [0]:
display(
    gold_dim_customer_scd
    .select(
        "CustomerKey",
        "FullName",
        "EffectiveStartDate",
        "EffectiveEndDate",
        "IsCurrent"
    )
    .limit(10)
)

### implement the actual SCD Type 2 change-detection logic for DimCustomer

In [0]:
from pyspark.sql.functions import current_date, lit

gold_dim_customer_scd = (
    gold_dim_customer
    .withColumn("EffectiveStartDate", current_date())
    .withColumn(
        "EffectiveEndDate",
        lit(None).cast("date")
    )
    .withColumn(
        "IsCurrent",
        lit(True)
    )
)

print("Initial SCD Type 2 DimCustomer prepared")
print("Rows:", gold_dim_customer_scd.count())

### Check Duplicate

In [0]:
display(
    gold_dim_customer_scd
    .groupBy("CustomerKey")
    .count()
    .filter("count > 1")
)

## Write the initial SCD Type 2 DimCustomer to Gold ADLS

In [0]:
gold_base = "abfss://gold@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025"

gold_dim_customer_scd.write \
    .mode("overwrite") \
    .parquet(f"{gold_base}/DimCustomer")

print("DimCustomer SCD Type 2 written to Gold successfully")

### validate

In [0]:
df_customer_gold_check = spark.read.parquet(
    f"{gold_base}/DimCustomer"
)

print("Rows:", df_customer_gold_check.count())

display(
    df_customer_gold_check.select(
        "CustomerKey",
        "FullName",
        "EffectiveStartDate",
        "EffectiveEndDate",
        "IsCurrent"
    ).limit(10)
)

## SCD Type 2 for DimProduct

In [0]:
from pyspark.sql.functions import current_date, lit

gold_dim_product_scd = (
    gold_dim_product
    .withColumn("EffectiveStartDate", current_date())
    .withColumn(
        "EffectiveEndDate",
        lit(None).cast("date")
    )
    .withColumn(
        "IsCurrent",
        lit(True)
    )
)

print("Initial SCD Type 2 DimProduct prepared")
print("Rows:", gold_dim_product_scd.count())

### Validate

In [0]:
display(
    gold_dim_product_scd.select(
        "ProductKey",
        "ProductAlternateKey",
        "EnglishProductName",
        "StartDate",
        "EndDate",
        "Status",
        "EffectiveStartDate",
        "EffectiveEndDate",
        "IsCurrent"
    ).limit(10)
)

### Check duplicate ProductKey

In [0]:
display(
    gold_dim_product_scd
    .groupBy("ProductKey")
    .count()
    .filter("count > 1")
)

## SCD Type 2 DimProduct to Gold

In [0]:
gold_base = "abfss://gold@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025"

gold_dim_product_scd.write \
    .mode("overwrite") \
    .parquet(f"{gold_base}/DimProduct")

print("DimProduct SCD Type 2 written to Gold successfully")

In [0]:
df_product_gold_check = spark.read.parquet(
    f"{gold_base}/DimProduct"
)

print("Rows:", df_product_gold_check.count())

display(
    df_product_gold_check.select(
        "ProductKey",
        "EnglishProductName",
        "EffectiveStartDate",
        "EffectiveEndDate",
        "IsCurrent"
    ).limit(10)
)

## SCD Type 2 for DimGeography

In [0]:
from pyspark.sql.functions import current_date, lit

gold_dim_geography_scd = (
    gold_dim_geography
    .withColumn("EffectiveStartDate", current_date())
    .withColumn(
        "EffectiveEndDate",
        lit(None).cast("date")
    )
    .withColumn(
        "IsCurrent",
        lit(True)
    )
)

print("Initial SCD Type 2 DimGeography prepared")
print("Rows:", gold_dim_geography_scd.count())

In [0]:
display(
    gold_dim_geography_scd.select(
        "GeographyKey",
        "City",
        "StateProvinceName",
        "EnglishCountryRegionName",
        "SalesTerritoryKey",
        "EffectiveStartDate",
        "EffectiveEndDate",
        "IsCurrent"
    ).limit(10)
)

In [0]:
display(
    gold_dim_geography_scd
    .groupBy("GeographyKey")
    .count()
    .filter("count > 1")
)

## Write DimGeo to Gold

In [0]:
gold_base = "abfss://gold@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025"

gold_dim_geography_scd.write \
    .mode("overwrite") \
    .parquet(f"{gold_base}/DimGeography")

print("DimGeography SCD Type 2 written to Gold successfully")

In [0]:
df_geography_gold_check = spark.read.parquet(
    f"{gold_base}/DimGeography"
)

print("Rows:", df_geography_gold_check.count())

display(
    df_geography_gold_check.select(
        "GeographyKey",
        "City",
        "EffectiveStartDate",
        "EffectiveEndDate",
        "IsCurrent"
    ).limit(10)
)

## Build Gold Fact FactInternetSales

In [0]:
gold_fact_internet_sales = df_fact.select(
    "ProductKey",
    "CustomerKey",
    "OrderDateKey",
    "DueDateKey",
    "ShipDateKey",
    "PromotionKey",
    "CurrencyKey",
    "SalesTerritoryKey",
    "RevisionNumber",
    "SalesOrderNumber",
    "SalesOrderLineNumber",
    "OrderQuantity",
    "UnitPrice",
    "UnitPriceDiscountPct",
    "DiscountAmount",
    "ProductStandardCost",
    "TotalProductCost",
    "SalesAmount",
    "TaxAmt",
    "Freight",
    "CarrierTrackingNumber",
    "CustomerPONumber"
)

display(gold_fact_internet_sales)

In [0]:
print(
    "FactInternetSales rows:",
    gold_fact_internet_sales.count()
)

### Check the fact business keys

In [0]:
display(
    gold_fact_internet_sales.select(
        "SalesOrderNumber",
        "SalesOrderLineNumber",
        "CustomerKey",
        "ProductKey",
        "OrderDateKey",
        "SalesAmount"
    ).limit(20)
)

### Check duplicate fact grain

In [0]:
display(
    gold_fact_internet_sales
    .groupBy(
        "SalesOrderNumber",
        "SalesOrderLineNumber"
    )
    .count()
    .filter("count > 1")
)

### CustomerKey validation

In [0]:
display(
    gold_fact_internet_sales
    .join(
        gold_dim_customer_scd.select("CustomerKey"),
        on="CustomerKey",
        how="left_anti"
    )
    .select("CustomerKey")
    .distinct()
)

### ProductKey validation

In [0]:
display(
    gold_fact_internet_sales
    .join(
        gold_dim_product_scd.select("ProductKey"),
        on="ProductKey",
        how="left_anti"
    )
    .select("ProductKey")
    .distinct()
)

### SalesTerritory validation

In [0]:
display(
    gold_fact_internet_sales
    .join(
        gold_dim_geography_scd.select("SalesTerritoryKey").distinct(),
        on="SalesTerritoryKey",
        how="left_anti"
    )
    .select("SalesTerritoryKey")
    .distinct()
)

### Date validation

In [0]:
date_keys = gold_dim_date.select(
    col("DateKey")
).distinct()

for key_column in ["OrderDateKey", "DueDateKey", "ShipDateKey"]:
    invalid_count = (
        gold_fact_internet_sales
        .join(
            date_keys,
            gold_fact_internet_sales[key_column] == date_keys["DateKey"],
            "left_anti"
        )
        .count()
    )

    print(f"{key_column}: {invalid_count} invalid keys")

## Write FactInternetSales

In [0]:
gold_base = "abfss://gold@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025"

gold_fact_internet_sales.write \
    .mode("overwrite") \
    .parquet(f"{gold_base}/FactInternetSales")

print("FactInternetSales written to Gold successfully")

### Verify

In [0]:
df_fact_gold_check = spark.read.parquet(
    f"{gold_base}/FactInternetSales"
)

print("Rows:", df_fact_gold_check.count())

display(df_fact_gold_check.limit(10))

## Write DimDate to Gold

In [0]:
gold_base = "abfss://gold@adventureworkstorageadls.dfs.core.windows.net/AdventureWorks2025"

gold_dim_date.write \
    .mode("overwrite") \
    .parquet(f"{gold_base}/DimDate")

print("DimDate written to Gold successfully")

## Gold API dataset

In [0]:
api_silver_path = "abfss://silver@adventureworkstorageadls.dfs.core.windows.net/API/Frankfurter"

df_api_silver = spark.read.parquet(api_silver_path)

display(df_api_silver)

In [0]:
from pyspark.sql.functions import col

gold_exchange_rate = df_api_silver.select(
    col("exchange_date").alias("ExchangeDate"),
    col("base_currency").alias("BaseCurrency"),
    col("INR_rate").alias("INRRate"),
    col("amount").alias("Amount")
)

display(gold_exchange_rate)

In [0]:
print("API Gold rows:", gold_exchange_rate.count())

gold_exchange_rate.printSchema()

In [0]:
gold_api_path = "abfss://gold@adventureworkstorageadls.dfs.core.windows.net/API/Frankfurter"

gold_exchange_rate.write \
    .mode("overwrite") \
    .parquet(gold_api_path)

print("Frankfurter API Gold data written successfully")